# Notebook 2 — Feature Engineering

Takes the clean output of Notebook 1 and builds every feature shared by both downstream tasks — calendar signals, cyclical time encoding, day-type energy baselines, and the `anomaly_flag` ground-truth label Task 2 needs.

**Every code block below has been run against the real dataset and verified — not just written.**

## Step 0 — Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("pandas:", pd.__version__)

pandas: 2.2.2


## Step 0b — Fix Working Directory

Same idempotent fix as Notebook 1 — safe to re-run any number of times.

In [2]:
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

Working directory: c:\Users\Dhruvajyoti Bordoloi\Desktop\smartdc-energy\ml-pipeline


## Step 1 — Load & Declare Input

In [3]:
INPUT_PATH = "data/processed/cleaned_timeseries.csv"
assert os.path.exists(INPUT_PATH), f"Missing input: {INPUT_PATH} — run 01_data_cleaning.ipynb first"

df = pd.read_csv(INPUT_PATH, parse_dates=['timestamp'])
print(df.shape)
df.head()

(26280, 11)


,timestamp,ambient_temp_c,relative_humidity_pct,chilled_water_temp_c,server_inlet_temp_c,under_floor_pressure_pa,it_load_kwh,cooling_load_kwh,support_load_kwh,total_power_kwh,pue
0,2022-01-01 00:00:00,-11.95,44.65,14.77,18.24,12.19,6377.75,964.32,166.72,7508.79,1.177342
1,2022-01-01 01:00:00,-11.80,44.09,14.71,18.20,11.92,6429.35,1142.50,183.65,7755.50,1.206265
2,2022-01-01 02:00:00,-13.50,44.32,14.65,17.02,12.49,6410.90,1107.80,179.33,7698.03,1.200772
3,2022-01-01 03:00:00,-15.18,46.93,14.97,17.17,11.30,6435.51,1233.69,154.72,7823.92,1.215742
4,2022-01-01 04:00:00,-17.99,48.08,14.91,17.11,12.86,6381.06,1153.06,156.78,7690.90,1.205270


## Step 2 — Calendar Features

In [4]:
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['month'] = df['timestamp'].dt.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# Sanity check against a KNOWN date — 2022-01-01 was a Saturday
known = df[df['timestamp'].dt.date == pd.Timestamp("2022-01-01").date()]
print("2022-01-01 is_weekend:", known['is_weekend'].iloc[0], "(should be 1)")

2022-01-01 is_weekend: 1 (should be 1)


## Step 3 — Cyclical Encoding

Raw hour/month integers create a false discontinuity (hour 23 to hour 0 looks like a huge jump). Sin/cos encoding fixes this — verified below that hour 23 and hour 0 land close together in the encoded space, while hour 23 and hour 11 (12 hours apart) land far apart.

In [5]:
df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)
df['month_sin'] = np.sin(2*np.pi*df['month']/12)
df['month_cos'] = np.cos(2*np.pi*df['month']/12)

h23 = df[df['hour']==23][['hour_sin','hour_cos']].iloc[0]
h0 = df[df['hour']==0][['hour_sin','hour_cos']].iloc[0]
h11 = df[df['hour']==11][['hour_sin','hour_cos']].iloc[0]
dist_near = np.sqrt((h23['hour_sin']-h0['hour_sin'])**2 + (h23['hour_cos']-h0['hour_cos'])**2)
dist_far = np.sqrt((h23['hour_sin']-h11['hour_sin'])**2 + (h23['hour_cos']-h11['hour_cos'])**2)
print(f"Hour 23 <-> Hour 0 distance: {dist_near:.3f} (should be small)")
print(f"Hour 23 <-> Hour 11 distance: {dist_far:.3f} (should be ~2.0, far)")
assert dist_near < dist_far, "Cyclical encoding phase check failed"

Hour 23 <-> Hour 0 distance: 0.261 (should be small)
Hour 23 <-> Hour 11 distance: 2.000 (should be ~2.0, far)


## Step 4 — Weekend / Midnight Energy Features

**Leakage-safe by design**: `.shift(1)` happens *before* `.rolling()`, so the feature never contains the current row's own value, even diluted — this is the exact fix for the bug caught earlier via a LightGBM feature-importance chart. `.ffill()` is used directly (not `fillna(method='ffill')`, which is fully removed in pandas 3.0). A final `.bfill()` resolves the single unavoidable edge case: the very first row of the entire 3-year series has no earlier value to pull from at all.

In [6]:
df['weekend_energy_kwh'] = df['total_power_kwh'].shift(1).where(df['is_weekend']==1)
df['weekend_energy_kwh'] = df['weekend_energy_kwh'].ffill().rolling(24*7, min_periods=1).mean().bfill()

midnight_mask = df['hour'].between(0,3)
df['midnight_energy_kwh'] = df['total_power_kwh'].shift(1).where(midnight_mask)
df['midnight_energy_kwh'] = df['midnight_energy_kwh'].ffill().rolling(24*7, min_periods=1).mean().bfill()

print("Nulls in weekend_energy_kwh:", df['weekend_energy_kwh'].isnull().sum())
print("Nulls in midnight_energy_kwh:", df['midnight_energy_kwh'].isnull().sum())

# Leakage check: correlation with current-row target should match correlation with the shifted target
corr_current = df['midnight_energy_kwh'].corr(df['total_power_kwh'])
corr_shifted = df['midnight_energy_kwh'].corr(df['total_power_kwh'].shift(1))
print(f"Corr with current target: {corr_current:.4f} | Corr with shifted target: {corr_shifted:.4f}")
print("(these should be nearly identical — if current is notably higher, leakage has crept back in)")

Nulls in weekend_energy_kwh: 0
Nulls in midnight_energy_kwh: 0
Corr with current target: 0.6480 | Corr with shifted target: 0.6483
(these should be nearly identical — if current is notably higher, leakage has crept back in)


## Step 5 — Anomaly Flag (Master Spec Rule 1/2)

A row is flagged `failure` if it breaches its physical buffer OR shows cross-column inconsistency (cooling unusually low while IT load is above-median — a cooling-strain signature). Thresholds are hand-set, not learned — disclosed here as a design choice, not hidden.

In [7]:
buffer_bounds = {
    'server_inlet_temp_c': (18, 30),
    'chilled_water_temp_c': (14.5, 24.5),
    'pue': (1.05, 1.38),
}
breach = pd.Series(False, index=df.index)
for col, (lo, hi) in buffer_bounds.items():
    col_breach = (df[col] < lo) | (df[col] > hi)
    print(f"{col}: {col_breach.sum()} rows breach buffer ({lo}-{hi})")
    breach |= col_breach

cross_inconsistent = (df['cooling_load_kwh'] < df['cooling_load_kwh'].quantile(0.05)) & \
                      (df['it_load_kwh'] > df['it_load_kwh'].quantile(0.5))
print(f"Cross-column inconsistency: {cross_inconsistent.sum()} rows")

df['anomaly_flag'] = np.where(breach | cross_inconsistent, 'failure', 'normal')
print()
print(df['anomaly_flag'].value_counts())
print(f"Anomaly rate: {(df['anomaly_flag']=='failure').mean()*100:.2f}%")

server_inlet_temp_c: 1008 rows breach buffer (18-30)
chilled_water_temp_c: 0 rows breach buffer (14.5-24.5)
pue: 0 rows breach buffer (1.05-1.38)
Cross-column inconsistency: 345 rows

anomaly_flag
normal     24959
failure     1321
Name: count, dtype: int64
Anomaly rate: 5.03%


## Step 6 — Visualize New Features

Mean power by hour should show a clear business-hours step pattern (confirming Step 3's cyclical features are phased correctly) — not flat, not inverted.

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
df.groupby('hour')['total_power_kwh'].mean().plot(ax=axes[0], marker='o', color='#3498db')
axes[0].set_title('Mean Power by Hour', fontweight='bold')
df['anomaly_flag'].value_counts().plot(kind='bar', ax=axes[1], color=['#2ecc71','#e74c3c'])
axes[1].set_title('Anomaly Flag Distribution', fontweight='bold')
plt.tight_layout(); 
plt.show()

<Figure size 1400x500 with 2 Axes>

## Step 7 — Save Output

This is the single file every downstream notebook (3, 4, 6) depends on.

In [9]:
OUTPUT_PATH = "data/processed/master_features.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {df.shape} to {OUTPUT_PATH}")

# Final verification before moving to Notebook 3/4/6
check = pd.read_csv(OUTPUT_PATH)
assert check.shape[0] == 26280, f"Wrong row count: {check.shape[0]}"
assert check.isnull().sum().sum() == 0, "Saved file still has nulls"
assert 'anomaly_flag' in check.columns, "anomaly_flag missing"
print(f"Verified: {check.shape[1]} columns, 0 nulls, ready for Notebooks 3, 4, and 6.")

Saved (26280, 22) to data/processed/master_features.csv
Verified: 22 columns, 0 nulls, ready for Notebooks 3, 4, and 6.
